# PDS 37 with VLTI/GRAVITY: data overview and telluric correction

PDS 37 was observed with **GRAVITY** at the Very Large Telescope Interferometer, which combines
light from four telescopes in the near-infrared K band (1.97–2.40 µm) at spectral resolution R ≈ 4000.
These observations resolve structure on milli-arcsecond scales.

This notebook:
1. inventories the GRAVITY exposures in `data/gravity/`,
2. applies and checks the telluric (Earth-atmosphere) correction of the spectra,
3. plots the interferometric observables in the continuum, around the Brackett-γ hydrogen line, and around the CO bandheads.

| Observable | What it tells us |
|---|---|
| `|V|` visibility amplitude | how resolved (large) the emission is on each baseline |
| `T3PHI` closure phase | departures from point symmetry; a non-zero signal is the signature of a binary or a lopsided disk |
| `DPHI` differential phase | shifts of the photocentre across a spectral line |
| `NFLUX` normalised flux | the spectrum, normalised to the continuum |

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pmoired

sys.path.insert(0, str(Path.cwd().parent))  # make the local `pds37` package importable
from pds37 import config

plt.rcParams["figure.dpi"] = 80
from astropy.io import fits
from pmoired import tellcorr
from pds37 import oifits

## 1. Data inventory

In [ ]:
gravity_paths = config.gravity_files()
inventory = oifits.summarize_files(gravity_paths).sort_values("mjd_start").reset_index(drop=True)
inventory

The observing dates and telescope stations are read from the FITS headers. Trust these over dates written into
file names, which are easy to get wrong.

## 2. Telluric correction

PMOIRED's `tellcorr.gravity` fits a model of Earth's atmospheric transmission to each spectrum and stores it
in a new `TELLURICS` extension inside the FITS file. That rewrites the file, so it is only run on files
that do not have the extension yet.

In [ ]:
def ensure_telluric_correction(path):
    """Run the GRAVITY telluric fit once per file. Returns True if it ran."""
    with fits.open(path) as hdul:
        if "TELLURICS" in hdul:
            return False
    tellcorr.gravity(str(path))
    return True


newly_corrected = [p.name for p in gravity_paths if ensure_telluric_correction(p)]
print(f"Corrected now: {newly_corrected or 'none'}; "
      f"already corrected: {len(gravity_paths) - len(newly_corrected)} file(s)")

In [ ]:
def plot_tellurics(path, ax=None):
    """Raw spectrum, fitted telluric transmission and corrected spectrum for one file."""
    ax = ax or plt.gca()
    with fits.open(path) as hdul:
        t = hdul["TELLURICS"].data
        wl = t["EFF_WAVE"] * 1e6
        ax.plot(wl, t["RAW_SPEC"], alpha=0.3, label="raw spectrum")
        ax.plot(wl, t["TELL_TRANS"] * np.mean(t["RAW_SPEC"]), "-y", alpha=0.6, label="telluric model")
        ax.plot(wl, t["RAW_SPEC"] / t["TELL_TRANS"], "-k", lw=0.8, label="corrected spectrum")
    ax.axvline(config.BRG_WL0, ls=":", color="tab:red", label="Brγ")
    ax.set(xlabel="wavelength (µm)", ylabel="flux (ADU)", title=path.name, ylim=(0, None))
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))


fig, ax = plt.subplots(figsize=(11, 4))
plot_tellurics(gravity_paths[0], ax)
fig.tight_layout()

## 3. Interferometric observables for one exposure

Change `EXAMPLE` to look at a different exposure. Closure phases that differ clearly from zero across the band
mean the source is not a single, point-symmetric star.

In [ ]:
EXAMPLE = 0  # index into gravity_paths
oi = pmoired.OI(str(gravity_paths[EXAMPLE]), insname=config.GRAVITY_INSNAME, verbose=False)

oi.setupFit({"obs": ["|V|", "T3PHI", "DPHI", "NFLUX"], "wl ranges": [config.K_BAND]})
oi.show()

### Zoom on Brackett-γ (2.166 µm)
The hydrogen emission line traces hot gas close to the star (accretion or wind). Changes in `|V|` and `DPHI` across the line would show that the line-emitting region has a different size or position from the continuum.

In [ ]:
oi.setupFit({"obs": ["|V|", "T3PHI", "DPHI", "NFLUX"], "wl ranges": [config.BRG_WINDOW]})
oi.show()

### Zoom on the CO bandheads (2.29–2.40 µm)

In [ ]:
oi.setupFit({"obs": ["|V|", "T3PHI", "DPHI", "NFLUX"], "wl ranges": [config.CO_BANDHEAD_WINDOW]})
oi.show()

## 4. Interactive explorer (run locally)

The widget below lets you flip through every exposure around Brγ. Widgets need a live Jupyter kernel,
so the plots will not appear in GitHub's static preview.

In [ ]:
import ipywidgets as widgets


def show_brg(name):
    obs = pmoired.OI(str(config.gravity_path(name)), insname=config.GRAVITY_INSNAME, verbose=False)
    obs.setupFit({"obs": ["|V|", "T3PHI", "DPHI", "NFLUX"], "wl ranges": [config.BRG_WINDOW]})
    obs.show()


widgets.interact(show_brg, name=widgets.Dropdown(options=[p.name for p in gravity_paths], description="File:"));